# Getting started with Tau

This notebook will run through the individual steps required to estimate copy number gain times in the evolution of a sample's tumor. It is intended for familiarizing yourself with how Tau works.

In [1]:
import sys, os

In [2]:
sys.path.append('../../package/')

In [3]:
import tau

### Input files

**Tau requires several input files to run:** \
a SNV VCF file (see `0bfd1043-7346-fdd0-e050-11ac0c484cab.consensus.20160830.somatic.snv_mnv.vcf.gz`) with all mutations \
a CNV text file (see `0bfd1043-7346-fdd0-e050-11ac0c484cab.consensus.20170119.somatic.cna.txt`) with copy number calls for each chromosomal region's major and minor alleles \
a purity text file (see `consensus.20170217.purity.ploidy.txt`) with tumor purity per sample \
an exposure text file (see `MuSiCal_assignment_PCAWG.txt`) with signature exposures per sample \
a signature CSV file (see `COSMIC_v3p2_SBS_WGS_MuSiCal.csv`) with trinucleotide context distributions per signature

In [4]:
sample = '0bfd1043-7346-fdd0-e050-11ac0c484cab'
snv_file = f'{sample}.consensus.20160830.somatic.snv_mnv.vcf.gz'
cnv_file = f'{sample}.consensus.20170119.somatic.cna.txt'
purity_file = 'consensus.20170217.purity.ploidy.txt'
output_dir = 'outputs'
output_file = os.path.join(output_dir, f'{sample}_trim.txt')

### Preprocessing

These various steps process the input CNV and SNV data such that our output can be used for estimating the timing of copy number gains in the evolution of a given sample's tumor \
\
Most crucially, these steps assign a multiplicity (how many copies of an allele the mutation is present on) to each SNV based on its VAF and the tumor purity of the sample

In [5]:
#if one specifies an output_file variable, it will be written to that file path. otherwise, nothing will be written
trim_df = tau.trim(sample, snv_file, cnv_file, purity_file, output_file)

Processing sample: 0bfd1043-7346-fdd0-e050-11ac0c484cab
SNV file: 0bfd1043-7346-fdd0-e050-11ac0c484cab.consensus.20160830.somatic.snv_mnv.vcf.gz
CNV file: 0bfd1043-7346-fdd0-e050-11ac0c484cab.consensus.20170119.somatic.cna.txt
Purity file: consensus.20170217.purity.ploidy.txt
Output file: outputs/0bfd1043-7346-fdd0-e050-11ac0c484cab_trim.txt


[W::bcf_hrec_check] Invalid tag name: "1000genomes_AF"
[W::bcf_hrec_check] Invalid tag name: "1000genomes_ID"
[W::bcf_hdr_register_hrec] The definition of Flag "INFO/dbsnp_somatic" is invalid, forcing Number=0
[E::idx_find_and_load] Could not retrieve index file for '0bfd1043-7346-fdd0-e050-11ac0c484cab.consensus.20160830.somatic.snv_mnv.vcf.gz'
[W::vcf_parse] Contig '1' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '10' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '11' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '12' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '13' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '14' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '15' is

Trim step completed successfully. Output saved to outputs/0bfd1043-7346-fdd0-e050-11ac0c484cab_trim.txt


/n/data1/hms/dbmi/park/jbrew/tools/miniforge3/envs/tau_new/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [6]:
categorize_df = tau.categorize(sample, df=trim_df)

In [7]:
revise_df, subclonal_CCF_df = tau.revise(sample, df=categorize_df)

Clustering subclones (Step 1)
Re-clustering subclones (Step 3)
Final subclone clustering (Step 4)


In [8]:
normalize_df = tau.normalize(sample, df=revise_df)

### Calculating signature likelihoods

This step assigns a signature to each mutation based on signature likelihood derived from pre-calculated exposures (e.g. from MuSiCal) and mutational signature distributions (e.g. from COSMIC), therefore enabling us to only use clock-like mutations (e.g. SBS1) in our timing estimates

In [9]:
exposure_file = 'MuSiCal_assignment_PCAWG.txt'
signature_file = 'COSMIC_v3p2_SBS_WGS_MuSiCal.csv'

In [10]:
likelihood_df = tau.sig_likelihoods(sample, df=normalize_df, 
                                    exposure_file=exposure_file, 
                                    signature_file=signature_file,
                                    ref_genome_path='/n/data1/hms/dbmi/park/jbrew/ref/hg19_decoy/human_g1k_v37_decoy.fasta')

sample: 0bfd1043-7346-fdd0-e050-11ac0c484cab


### Calculating mutation multiplicity counts

This step counts the number of mutations per multiplicity, e.g. the number of mutations present 1 copy, 2 copies, 3 copies, etc. such that we have the required output format to estimate the timing of copy number gains

In [6]:
multiplicities_output= f'outputs/{sample}_multiplicities.txt'

In [ ]:
multiplicities = tau.count_multiplicities(df=likelihood_df, output_file=multiplicities_output)

Results saved to outputs/0bfd1043-7346-fdd0-e050-11ac0c484cab_multiplicities.txt


### Solving for timing solutions

This function will calculate the timing solutions and plot the result, either to a file (output_plot) or just to the notebook if no `output_plot` is not specified. `average` will determine whether the plot shows averaged timings or ranges.

In [7]:
segments, solutions_df, breakpoints = tau.calculate_timing_solutions(sample, 
                                                                        multiplicities_file=multiplicities_output,
                                                                        average=False, 
                                                                        output_plot=f'outputs/plots/{sample}_timing_plot.pdf')

In [15]:
import matplotlib.pyplot as plt

In [20]:
breaks = [x for y in breakpoints for x in y]